# Baybayin OCR Training (Google Colab Edition)

This notebook mirrors the shell pipeline we used earlier, but is tailored for Google Colab so you can train on a public Kaggle dataset or files stored on Drive. The general flow is:

1. Install dependencies (Tesseract, tesstrain prerequisites, Python libs).
2. Clone this project so we can reuse the existing scripts (`adjust_box_coordinates.py`, `run_training.sh`, etc.).
3. Bring your dataset into the runtime (via Kaggle CLI, Google Drive, or manual upload).
4. Optionally tighten `.box` coordinates.
5. Launch the training pipeline with `run_training.sh`.
6. Summarise the generated `training.log`.
7. Evaluate the trained model on holdout data and generate prediction visuals.

> ⚠️ **Note:** Training on large datasets (tens of thousands of samples) can take a long time and may exceed Colab limits. You can start with a smaller subset to verify everything works.

In [ ]:
!nvidia-smi

## 1. Install Dependencies
We need Tesseract (and its headers), ImageMagick (for DPI adjustments), plus a few Python packages. This cell may take a few minutes.

In [ ]:
%%bash
set -e
apt-get -qq update
apt-get -qq install -y tesseract-ocr libtesseract-dev imagemagick git
pip install -q numpy pillow kaggle

## 2. Clone the Training Utilities
Set `GIT_REPO_URL` to the repository that contains `run_training.sh`, `adjust_box_coordinates.py`, and the helper scripts. By default it points to this working copy; feel free to fork and change the URL.

In [ ]:
GIT_REPO_URL = "https://github.com/<YOUR_GITHUB_USERNAME>/baybayin_project.git"
PROJECT_DIR = "/content/baybayin_project"
print("Repository URL:", GIT_REPO_URL)
print("Target directory:", PROJECT_DIR)

In [ ]:
import os
if not os.path.exists(PROJECT_DIR):
    !git clone $GIT_REPO_URL $PROJECT_DIR
else:
    print("Repository already cloned at", PROJECT_DIR)

## 3. (Optional) Download Dataset via Kaggle CLI
If your dataset is on Kaggle, upload your `kaggle.json` once, then download & unzip. Otherwise, mount Google Drive and copy files into `/content`.

> Replace `<DATASET_SLUG>` with the actual `owner/dataset-name`.

In [ ]:
# from google.colab import files
# files.upload()  # upload kaggle.json once
import pathlib
from pathlib import Path

if Path("kaggle.json").exists():
    Path("~/.kaggle").expanduser().mkdir(exist_ok=True)
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
else:
    raise FileNotFoundError("Upload kaggle.json above or skip this cell if using Drive uploads.")

KAGGLE_DATASET = "<DATASET_SLUG>"  # e.g. "username/baybayin-dataset"
RAW_DATA_DIR = "/content/baybayin_data"

!mkdir -p $RAW_DATA_DIR
!kaggle datasets download -d $KAGGLE_DATASET -p $RAW_DATA_DIR --unzip

If you're using Google Drive instead, uncomment and run the following cell, then set `RAW_DATA_DIR` to the folder you mounted.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# RAW_DATA_DIR = "/content/drive/MyDrive/path-to-your-dataset"

## 4. Define Paths and Parameters
Set the dataset directory, model name, and other knobs here. The dataset should already contain `.tif`, `.box`, and `.gt.txt` files.

In [ ]:
DATASET_FOR_TRAINING = RAW_DATA_DIR  # change if you reorganise files
MODEL_NAME = "bay_full_corrected"
MAX_ITERATIONS = 10000
TESSDATA_DIR = os.path.join(PROJECT_DIR, "tesseract_training", "data")
HOLDOUT_SOURCE = RAW_DATA_DIR  # path with extra samples for evaluation
HOLDOUT_VIS_DIR = "/content/holdout_visuals"

for key, value in {
    "DATASET_FOR_TRAINING": DATASET_FOR_TRAINING,
    "MODEL_NAME": MODEL_NAME,
    "MAX_ITERATIONS": MAX_ITERATIONS,
    "TESSDATA_DIR": TESSDATA_DIR,
}.items():
    print(f"{key}: {value}")

## 5. Optionally Tighten Bounding Boxes
This overwrites `.box` files so they tightly hug each glyph.

In [ ]:
%%bash
set -e
cd /content/baybayin_project
python3 adjust_box_coordinates.py --dataset "$DATASET_FOR_TRAINING" --min-pixels 4

## 6. Launch Training
Now we're ready to start the Tesstrain pipeline. This can take quite a while!

In [ ]:
%%bash
set -e
cd /content/baybayin_project
MODEL_NAME="$MODEL_NAME" \
SOURCE_DATASET="$DATASET_FOR_TRAINING" \
MAX_ITERATIONS="$MAX_ITERATIONS" \
./run_training.sh

## 7. Summarise `training.log`
Convert the long log into a concise JSON summary of key checkpoints.

In [ ]:
import json
import pathlib
import re

log_path = pathlib.Path(PROJECT_DIR) / "tesseract_training" / "data" / MODEL_NAME / "training.log"
print("Log path:", log_path)
if not log_path.exists():
    raise FileNotFoundError("training.log not found – did training finish?")

pattern = re.compile(r"At iteration (\d+)/(\d+)/(\d+), Mean rms=([0-9.]+)%, delta=([0-9.]+)%, char train=([0-9.]+)%, word train=([0-9.]+)%, skip ratio=([0-9.]+)%")
checkpoints = {}
for line in log_path.read_text().splitlines():
    match = pattern.search(line)
    if not match:
        continue
    iteration = int(match.group(1))
    bucket = max(1000, (iteration // 1000) * 1000)
    if bucket not in checkpoints or iteration < checkpoints[bucket]["iteration"]:
        checkpoints[bucket] = {
            "iteration": iteration,
            "mean_rms": float(match.group(4)),
            "delta": float(match.group(5)),
            "char_error": float(match.group(6)),
            "word_error": float(match.group(7)),
        }

summary = [{"bucket": bucket, **data} for bucket, data in sorted(checkpoints.items())]
print(json.dumps(summary, indent=2))

## 8. Evaluate on Holdout Samples
Use images that were not in the training set to gauge real performance.

In [ ]:
%%bash
cd /content/baybayin_project
python3 evaluate_holdout.py \
  --full-set "$HOLDOUT_SOURCE" \
  --train-set "$DATASET_FOR_TRAINING" \
  --tessdata-dir "$TESSDATA_DIR" \
  --model "$MODEL_NAME" \
  --limit 200

## 9. Generate Visualisations
These PNGs overlay bounding boxes and include ground-truth vs. prediction text for manual inspection.

In [ ]:
%%bash
cd /content/baybayin_project
python3 generate_holdout_visuals.py \
  --full-set "$HOLDOUT_SOURCE" \
  --train-set "$DATASET_FOR_TRAINING" \
  --tessdata-dir "$TESSDATA_DIR" \
  --model "$MODEL_NAME" \
  --limit 10 \
  --offset 0 \
  --output-dir "$HOLDOUT_VIS_DIR"

## 10. View Outputs
- Trained model: `tesseract_training/data/<MODEL_NAME>.traineddata`
- Training logs & checkpoints: `tesseract_training/data/<MODEL_NAME>/`
- Holdout visualisations: `HOLDOUT_VIS_DIR`

You can download the trained model using Colab's file browser or via `files.download`.

In [ ]:
from google.colab import files
import pathlib
traineddata_path = pathlib.Path(PROJECT_DIR) / "tesseract_training" / "data" / f"{MODEL_NAME}.traineddata"
if traineddata_path.exists():
    files.download(str(traineddata_path))
else:
    print("No traineddata file found – ensure training finished successfully.")